<a href="https://colab.research.google.com/github/KA18202005/DeepLearning-Learning/blob/main/Age_Gender_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [3]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
 90% 298M/331M [00:00<00:00, 843MB/s] 
100% 331M/331M [00:00<00:00, 790MB/s]


In [4]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip",'r')
zip.extractall("/content")
zip.close()

In [5]:
import os
import numpy as np
import pandas as pd
import tensorflow
# from tensorflow.keras.preprocessing.image import ImageDataGenerator # This is deprecated in Keras 3.x

In [6]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [7]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [8]:
len(age)

23708

In [9]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [10]:
df.shape

(23708, 3)

In [11]:
df.head()

,age,gender,img
0,38,1,38_1_1_20170113005018855.jpg.chip.jpg
1,31,1,31_1_2_20170105161436755.jpg.chip.jpg
2,54,0,54_0_1_20170113151911864.jpg.chip.jpg
3,74,0,74_0_2_20170105174417462.jpg.chip.jpg
4,25,0,25_0_0_20170117151647102.jpg.chip.jpg


In [12]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [13]:
train_df.shape

(20000, 3)

In [14]:
test_df.shape

(3708, 3)

In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img

In [16]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [17]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [18]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [19]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [20]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

# Unfreeze some layers of ResNet50 for fine-tuning
for layer in resnet.layers:
    if not isinstance(layer, BatchNormalization):
        layer.trainable = False
# Let's unfreeze the last few convolutional blocks for fine-tuning
for layer in resnet.layers[-30:]:
    if not isinstance(layer, BatchNormalization):
        layer.trainable = True


output = resnet.layers[-1].output

flatten = Flatten()(output)

# Adding Dropout for regularization
dense1 = Dense(512, activation='relu')(flatten)
dropout1 = Dropout(0.5)(dense1)
dense2 = Dense(512,activation='relu')(flatten)
dropout2 = Dropout(0.5)(dense2)

dense3 = Dense(512,activation='relu')(dropout1)
dropout3 = Dropout(0.5)(dense3)
dense4 = Dense(512,activation='relu')(dropout2)
dropout4 = Dropout(0.5)(dense4)

output1 = Dense(1,activation='linear',name='age')(dropout3)
output2 = Dense(1,activation='sigmoid',name='gender')(dropout4)

In [21]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [22]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':5,'gender':5})

In [23]:
def multi_output_generator_wrapper(generator):
    for x, y in generator:
        yield x, tuple(y)



In [24]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

BATCH_SIZE = 32

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

model.fit(
    multi_output_generator_wrapper(train_generator),
    steps_per_epoch=len(train_df) // BATCH_SIZE,
    epochs=10, # Increased epochs as early stopping will manage overfitting
    validation_data=multi_output_generator_wrapper(test_generator),
    validation_steps=len(test_df) // BATCH_SIZE,
    callbacks=[early_stopping, model_checkpoint]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 335s 457ms/step - age_loss: 13.8076 - age_mae: 13.8076 - gender_accuracy: 0.6596 - gender_loss: 1.7541 - loss: 77.8085 - val_age_loss: 24.1652 - val_age_mae: 24.1652 - val_gender_accuracy: 0.4769 - val_gender_loss: 1.5879 - val_loss: 128.7658
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 291s 467ms/step - age_loss: 9.3913 - age_mae: 9.3913 - gender_accuracy: 0.7971 - gender_loss: 0.4765 - loss: 49.3389 - val_age_loss: 9.3847 - val_age_mae: 9.3847 - val_gender_accuracy: 0.8712 - val_gender_loss: 0.3092 - val_loss: 48.4697
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 269s 430ms/step - age_loss: 8.7644 - age_mae: 8.7644 - gender_accuracy: 0.8235 - gender_loss: 0.4113 - loss: 45.8783 - val_age_loss: 12.6557 - val_age_mae: 12.6574 - val_gender_accuracy: 0.8419 - val_gender_loss: 0.3219 - val_loss: 64.8962
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 254s 406ms/step - age_loss: 8.1895 - age_mae: 8.1895 - gender_accuracy: 0.8391 - gender_loss: 0.3530 - loss: 42.7125 - 

In [34]:
import cv2

In [35]:
test_img = cv2.imread('/content/man.jpg')

In [36]:
test_img = cv2.resize(test_img, (200, 200))

In [43]:
test_input = test_img.reshape((1, 200, 200, 3)) / 255.0

In [45]:
model.predict(test_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step


[array([[39.976727]], dtype=float32), array([[0.23612535]], dtype=float32)]

In [39]:
test_img1 = cv2.imread('/content/woman.jpg')

In [40]:
test_img1 = cv2.resize(test_img1, (200, 200))

In [44]:
test_input1 = test_img1.reshape((1, 200, 200, 3)) / 255.0

In [46]:
model.predict(test_input1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


[array([[22.396511]], dtype=float32), array([[0.86368126]], dtype=float32)]

First, you need to install the `streamlit` library. If you haven't already, make sure `opencv-python-headless` is also installed, as it's needed for image processing without a GUI environment.

In [47]:
!pip install streamlit opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 102.6 MB/s eta 0:00:00


Next, let's create a Python script named `app.py` that will contain your Streamlit application code. This script will:

1.  Load the trained Keras model.
2.  Provide an interface to upload an image.
3.  Preprocess the uploaded image to match the model's input requirements.
4.  Make predictions (age and gender).
5.  Display the uploaded image and the prediction results.

In [48]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import cv2
from PIL import Image
import os

# Load the model
# Make sure the 'best_model.keras' or 'age_gender_model.keras' exists in your Colab environment
@st.cache_resource
def get_model():
    model_path = 'best_model.keras' # Or 'age_gender_model.keras' if you explicitly saved it
    if not os.path.exists(model_path):
        st.error(f"Model file not found at {model_path}. Please ensure it's saved correctly.")
        return None
    model = load_model(model_path)
    return model

model = get_model()

st.title('Age and Gender Prediction')
st.write('Upload an image and let the model predict age and gender.')

if model is None:
    st.stop()

uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Read the image file
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Uploaded Image.', use_column_width=True)
    st.write("")
    st.write("Classifying...")

    # Preprocess the image
    img_array = np.array(image)
    img_resized = cv2.resize(img_array, (200, 200)) # Model input size
    img_normalized = img_resized.reshape((1, 200, 200, 3)) / 255.0

    # Make prediction
    predictions = model.predict(img_normalized)
    age_prediction = predictions[0][0][0]
    gender_prediction_prob = predictions[1][0][0]

    # Determine gender label
    # Assuming 0: Male, 1: Female from your training data encoding
    gender_label = 'Female' if gender_prediction_prob > 0.5 else 'Male'

    st.subheader(f'Predicted Age: {int(age_prediction)} years old')
    st.subheader(f'Predicted Gender: {gender_label} (Probability: {gender_prediction_prob:.2f})')


Writing app.py


Now that you have your `app.py` file, you can run your Streamlit application directly from Google Colab.

Streamlit apps run on a local server, but Colab provides a way to expose them via `ngrok` or `localtunnel` (which `streamlit run` often handles automatically).

Run the following command in a new code cell to start your Streamlit app. It will provide a public URL you can click to access your web application.

In [49]:
!streamlit run app.py &>/dev/null&  # Run in background
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://metal-spies-wear.loca.lt
^C
